# 제출용 노트북 — 앙상블 (베이스 + v2), 중간 저장 지원

## 이 노트북의 목적
**8/28 리더보드 검증과 8/31 최종 제출에 같은 노트북을 씁니다.**
`TARGET`만 바꾸면 리더보드(831문제)와 최종 test로 전환됩니다.

## ★ 중간 저장이 핵심입니다

831문제 x 64샘플 = 약 6시간 30분입니다.
지금까지 세션 끊김을 4번, Ctrl+Z 사고를 1번 겪었습니다. 한 번에 다 돌리는 건 위험합니다.

→ **문제를 덩어리(100문제)로 나눠 각 결과를 `/kaggle/working/`에 저장**합니다.
→ 끊기면 **이미 끝난 덩어리는 건너뛰고 이어서** 돌립니다.
→ 셀을 다시 실행해도 안전합니다(멱등).

## 규칙 확인

| 규칙 | 이 노트북 | 판정 |
|---|---|---|
| 4.1a 베이스 모델 고정 | Qwen2.5-3B-Instruct만 사용 | ○ |
| 4.3 외부 모델 앙상블 금지 | 동일 베이스 + 자체 LoRA 어댑터 1개 | ○ (8/23 주최 측 확인) |
| 유형별 라우팅 금지 | 없음 — 모든 문제에 균일 적용 | ○ |
| 추론 시 인터넷·도구 호출 금지 | 모델 출력만 사용, 코드 실행 없음 | ○ |
| 5.2c 외부 데이터 목록 명시 | **제출 시 NuminaMath-1.5 기재** | ▲ 잊지 말 것 |

## ★ 실행 방법 — Save & Run All (커밋)

1. Settings: **Accelerator = GPU T4 x2**, **Internet = On**
2. Input: 대회 데이터 + `qwen25-3b-v2-lora` (**어댑터는 이것 하나만**)
3. `[1]`에서 `TARGET` 확인
4. 우측 상단 **Save Version → Save & Run All (Commit)**
5. **브라우저 닫고 나가셔도 됩니다.** 서버가 알아서 끝냅니다
6. 완료 후 버전 페이지 **Output**에서 `submission.csv` 다운로드

셀을 하나씩 눌러도 되지만, **6시간 넘는 작업은 반드시 커밋으로** 하세요.

## 검증된 근거
| 조합 | 로컬 maj@32 |
|---|---|
| 베이스 단독 | 0.7400 |
| **베이스32 + v2_32** | **0.7500** |

리더보드 환산 예상 0.792 (현재 0.78580)


---
## [1] 설정 ▶️ 항상 실행

### `TARGET`
- `"leaderboard"` — 831문제. 8/28 검증용
- `"test"` — 8/31 공개되는 최종 test. **그날 이것만 바꾸면 됩니다**

### `CHUNK`
한 번에 처리할 문제 수. 100이면 831문제가 9덩어리로 나뉩니다.
덩어리마다 약 43분 걸리고, 끝나면 즉시 파일로 저장됩니다.
작을수록 로그에 진행 상황이 자주 남고, 대화형으로 돌릴 때 이어받기 손해가 줄어듭니다.

### `USE_V2` / `N_BASE` / `N_V2`
`USE_V2=False`로 두면 베이스 단독 N=32(기존 0.78580 설정)로 돌아갑니다.
앙상블이 리더보드에서 안 통하면 이 스위치로 되돌리세요.

In [ ]:
TARGET     = "leaderboard"   # "leaderboard" | "test"
USE_V2     = True            # False면 베이스 단독
N_BASE     = 32              # 베이스 샘플 수
N_V2       = 32              # v2 샘플 수 (USE_V2=True일 때)
TEMP       = 0.8
MAX_TOKENS = 1024
CHUNK      = 100             # 중간 저장 단위 (덩어리당 약 43분)
SEED       = 42
LORA_RANK  = 64
MODEL_ID   = "Qwen/Qwen2.5-3B-Instruct"
WORK       = "/kaggle/working"
SYSTEM = ("You are an expert competition mathematician. Solve the problem step by step, "
          "concisely. The final answer is ALWAYS a single integer. "
          "End your response with the final integer inside \\boxed{}.")
print(f"TARGET={TARGET} | base {N_BASE} + v2 {N_V2 if USE_V2 else 0} = {N_BASE + (N_V2 if USE_V2 else 0)}표/문제")

---
## [2] 설치 ▶️ ★ 커밋 모드용 — 재시작 불필요

### 왜 이 순서인가
Kaggle 커널은 부팅하면서 **protobuf 5.29.5를 이미 메모리에 올려둡니다.** 우리 코드가 실행되기 전에요.
그래서 디스크를 6.x로 올려도 메모리는 5.x라 충돌합니다.

→ **디스크를 메모리에 맞춥니다.** vLLM을 먼저 깔고(protobuf 6.x가 딸려 옴),
`--force-reinstall`로 5.29.5를 덮어씁니다.

`ray`와 `opentelemetry`가 protobuf 6.x용 코드를 들고 오는 범인이었습니다.
둘 다 GPU 한 장 추론에는 필요 없고, 없으면 vLLM이 알아서 건너뜁니다.

### 이게 왜 중요한가
**중간 재시작이 필요 없으므로 Save & Run All(커밋)이 가능합니다.**

> ⚠️ 대화형 세션은 유휴 상태가 되면 "Are you still here?"가 뜨고,
> **Continue를 누르면 커널이 재시작**되어 실행 중이던 작업이 죽습니다.
> 6시간 넘는 작업을 화면 앞에서 지킬 수는 없으므로 **커밋이 유일한 방법**입니다.

In [ ]:
!pip install -q -U vllm 2>&1 | tail -1
!pip uninstall -q -y ray opentelemetry-exporter-otlp-proto-grpc opentelemetry-exporter-otlp
!pip install -q --force-reinstall "protobuf==5.29.5" 2>&1 | tail -1

import google.protobuf as p, torch
print("protobuf:", p.__version__)
print("GPU    :", torch.cuda.device_count())
assert torch.cuda.device_count() > 0, "GPU 없음 — Settings에서 Accelerator를 켜세요"
from vllm import LLM
print("vllm import 성공 — 재시작 불필요")

---
## [3] 데이터 로드 ▶️

`TARGET`에 따라 대상을 고릅니다. 8/31에는 `test` 파일이 `/kaggle/input/`에 추가되므로
자동 탐색이 알아서 찾습니다.

⚠️ 어댑터 Dataset은 **`qwen25-3b-v2-lora` 하나만** 붙어 있어야 합니다.

In [ ]:
import glob, os, pandas as pd

def find_csv(must_have, must_not=()):
    for p in sorted(glob.glob("/kaggle/input/**/*.csv", recursive=True)):
        b = os.path.basename(p).lower()
        if all(k in b for k in must_have) and not any(k in b for k in must_not):
            return p

if TARGET == "leaderboard":
    PATH = find_csv(["leaderboard", "filtered"]) or find_csv(["leaderboard"])
else:
    PATH = find_csv(["test"], must_not=["latest"])
assert PATH, f"{TARGET} 파일을 못 찾았습니다"
print("대상 파일:", PATH)

work = pd.read_csv(PATH)
print(f"문제 수: {len(work):,}")
print("컬럼:", list(work.columns))
assert "id" in work.columns and "question" in work.columns

LORA_PATH = None
if USE_V2:
    cfgs = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
    assert len(cfgs) == 1, f"어댑터가 {len(cfgs)}개 발견됨. 정확히 1개여야 합니다: {cfgs}"
    LORA_PATH = os.path.dirname(cfgs[0])
    assert "v2" in LORA_PATH.lower(), f"v2 어댑터가 아닙니다: {LORA_PATH}"
    print("어댑터:", LORA_PATH)

n_chunk = (len(work) + CHUNK - 1) // CHUNK
print(f"\n{n_chunk}개 덩어리로 분할 (덩어리당 최대 {CHUNK}문제)")
print(f"예상 소요: 약 {len(work)*(N_BASE+(N_V2 if USE_V2 else 0))*0.315/3600:.1f}시간")

---
## [4] 답 추출기 ▶️

In [ ]:
import re
from collections import Counter

def extract_boxed(text):
    """Return the raw content inside the LAST \\boxed{...}, brace-balanced."""
    idx = text.rfind('\\boxed')
    if idx == -1:
        return None
    i = idx + len('\\boxed')
    while i < len(text) and text[i] == ' ':
        i += 1
    if i >= len(text):
        return None
    if text[i] != '{':                       # bare form: \boxed 15
        m = re.match(r'-?[\d,]+', text[i:])
        return m.group(0) if m else None
    depth, start = 0, i + 1
    while i < len(text):
        if text[i] == '{':
            depth += 1
        elif text[i] == '}':
            depth -= 1
            if depth == 0:
                return text[start:i]
        i += 1
    return None

def to_int(s):
    """LaTeX/text -> python int, or None. Never uses float(), so huge ints survive."""
    if s is None:
        return None
    s = str(s).strip()
    s = s.replace('{,}', '').replace('{\\,}', '')          # LaTeX thousands separator
    s = re.sub(r'\\(?:text|mathrm|mbox|textbf|textrm)\s*\{([^{}]*)\}', r'\1', s)
    for junk in ['\\!', '\\,', '\\;', '\\:', '\\ ', '\\left', '\\right',
                 '\\$', '$', '%', '~', '^\\circ', '\\%']:
        s = s.replace(junk, '')
    s = s.replace(',', '').replace(' ', '').strip()
    s = re.sub(r'[a-zA-Z]+$', '', s)                       # trailing unit: 42cm -> 42
    while len(s) > 1 and s[0] == '(' and s[-1] == ')':     # (\frac{100}{4}) -> \frac{100}{4}
        s = s[1:-1].strip()
    s = s.rstrip('.')
    if not s:
        return None
    m = re.fullmatch(r'\\[dt]?frac\{([-+]?\d+)\}\{([-+]?\d+)\}', s)
    if m:
        a, b = int(m.group(1)), int(m.group(2))
        return a // b if b != 0 and a % b == 0 else None
    m = re.fullmatch(r'([-+]?\d+)/([-+]?\d+)', s)
    if m:
        a, b = int(m.group(1)), int(m.group(2))
        return a // b if b != 0 and a % b == 0 else None
    m = re.fullmatch(r'([-+]?\d+)(?:\\times|\\cdot)10\^\{?(\d+)\}?', s)
    if m:
        return int(m.group(1)) * 10 ** int(m.group(2))
    if re.fullmatch(r'[-+]?\d+', s):
        return int(s)
    m = re.fullmatch(r'([-+]?\d+)\.0*', s)
    if m:
        return int(m.group(1))
    return None

def last_int(text):
    for c in reversed(re.findall(r'-?\d[\d,]*', text)):
        v = to_int(c)
        if v is not None:
            return v
    return None

def parse_answer(text):
    """None means 'this sample produced no usable integer' -> dropped from voting."""
    raw = extract_boxed(text)
    if raw is not None:
        return to_int(raw)          # boxed present but unparseable -> None, do NOT guess
    m = re.findall(r'(?:answer|Answer|ANSWER)\s*(?:is|:|=)+\s*\$?(-?[\d,]+)', text)
    if m:
        v = to_int(m[-1])
        if v is not None:
            return v
    return last_int(text)

def majority_vote(values, fallback=0):
    vals = [v for v in values if v is not None]
    if not vals:
        return fallback
    return Counter(vals).most_common(1)[0][0]

_c = [(r"\boxed{132}",132), (r"\boxed{-2,025,078}",-2025078),
      (r"\boxed{\dfrac{650}{5}}",130), (r"\boxed{42 \text{ cm}}",42),
      (r"\boxed{\frac{7}{2}}",None), (r"no box, ends with 12 cats",12)]
bad = sum(parse_answer(t) != w for t, w in _c)
print("parser FAILURES:", bad, "/", len(_c))
assert bad == 0, "파서 자체 검증 실패 — 진행하지 마세요"

---
## [5] 모델 로드 ▶️ 5~10분

`enable_lora=True`로 올리면 한 세션에서 어댑터를 붙였다 뗐다 할 수 있습니다.
`USE_V2=False`여도 이 옵션은 켜둡니다(성능 영향 없음).

⚠️ 대화형으로 돌릴 때 **이 셀을 두 번 실행하면 GPU 메모리 에러**가 납니다. 재실행하려면 세션 재시작.

In [ ]:
import time, json
import numpy as np
from collections import Counter
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

llm = LLM(model=MODEL_ID, dtype="half", max_model_len=4096,
          gpu_memory_utilization=0.90, tensor_parallel_size=1,
          seed=SEED, trust_remote_code=True,
          enable_lora=True, max_lora_rank=LORA_RANK)

from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_ID)

def make_prompts(questions):
    return [tok.apply_chat_template(
        [{"role":"system","content":SYSTEM},{"role":"user","content":q}],
        tokenize=False, add_generation_prompt=True) for q in questions]

print("로드 완료")

---
## ★ [6] 덩어리별 생성 + 중간 저장 ▶️ 가장 오래 걸림

### 어떻게 동작하나
```
덩어리 0 (문제 0~199)   → 생성 → chunk_000.json 저장
덩어리 1 (문제 200~399) → 생성 → chunk_001.json 저장
...
```

**이미 저장된 덩어리는 건너뜁니다.** 세션이 끊겨서 다시 실행해도 남은 것만 돌립니다.

### 끊겼을 때
1. 세션 재시작 → `[1]` `[3]` `[4]` `[5]` 실행
2. **이 셀 다시 실행** — 저장된 덩어리는 자동으로 건너뜁니다

⚠️ **커밋 모드에서는 처음부터 끝까지 한 번에 돌므로 이어받기가 쓰일 일이 없습니다.**
덩어리 저장은 (a) 로그에 진행 상황을 남기고 (b) 대화형으로 돌릴 때 이어받기를 가능하게 하는 용도입니다.

대화형으로 돌리는 경우: Restart Session은 견디지만 **Stop을 누르면 `/kaggle/working/`이 사라집니다.**

### 진행 상황
덩어리마다 소요 시간과 누적 진행률이 출력됩니다.

In [ ]:
os.makedirs(f"{WORK}/chunks", exist_ok=True)

def chunk_path(i):
    return f"{WORK}/chunks/chunk_{i:03d}.json"

t_start = time.time()
for ci in range(n_chunk):
    cp = chunk_path(ci)
    if os.path.exists(cp):
        print(f"[{ci+1}/{n_chunk}] 이미 저장됨 — 건너뜀")
        continue

    sub = work.iloc[ci*CHUNK : (ci+1)*CHUNK]
    prompts = make_prompts(sub["question"])
    t0 = time.time()

    # 베이스
    sp_b = SamplingParams(n=N_BASE, temperature=TEMP, top_p=0.95,
                          max_tokens=MAX_TOKENS, seed=SEED)
    outs_b = llm.generate(prompts, sp_b, use_tqdm=False)
    vals = [[parse_answer(c.text) for c in o.outputs] for o in outs_b]

    # v2 어댑터
    if USE_V2:
        sp_v = SamplingParams(n=N_V2, temperature=TEMP, top_p=0.95,
                              max_tokens=MAX_TOKENS, seed=SEED)
        outs_v = llm.generate(prompts, sp_v, use_tqdm=False,
                              lora_request=LoRARequest("v2", 1, LORA_PATH))
        vals = [b + [parse_answer(c.text) for c in o.outputs]
                for b, o in zip(vals, outs_v)]

    with open(cp, "w") as f:
        json.dump({"ids": list(sub["id"]), "vals": vals}, f)

    el = time.time() - t0
    done = ci + 1
    print(f"[{done}/{n_chunk}] {len(sub)}문제 {el/60:.1f}분 저장 완료 "
          f"| 누적 {(time.time()-t_start)/60:.0f}분 "
          f"| 남은 예상 {(n_chunk-done)*el/60:.0f}분")

print(f"\n전체 생성 완료: {(time.time()-t_start)/60:.1f}분")

---
## [7] 다수결 + submission.csv ▶️ (GPU 미사용)

저장된 덩어리를 모두 읽어 합칩니다. **이 셀은 몇 번 실행해도 안전합니다.**

### 확인할 것
- 행 수가 문제 수와 일치
- 컬럼명이 **소문자 `id`** (대문자면 채점기가 거부)
- 파싱 실패율 (기준: 베이스 단독 1.70%, v2 앙상블 2.1% 예상)

In [ ]:
def majority(vals, fb=0):
    v = [x for x in vals if x is not None]
    return Counter(v).most_common(1)[0][0] if v else fb

ids, preds, n_fail, n_tot, shares = [], [], 0, 0, []
for ci in range(n_chunk):
    cp = chunk_path(ci)
    assert os.path.exists(cp), f"덩어리 {ci}가 없습니다 — [6]을 다시 실행하세요"
    d = json.load(open(cp))
    for _id, vals in zip(d["ids"], d["vals"]):
        ids.append(_id)
        preds.append(majority(vals))
        n_fail += sum(v is None for v in vals)
        n_tot  += len(vals)
        cnt = Counter([v for v in vals if v is not None])
        shares.append(cnt.most_common(1)[0][1]/len(vals) if cnt else 0)

print(f"문제 {len(ids):,} | 샘플 {n_tot:,}")
print(f"파싱 실패율 {n_fail/n_tot:.2%} | 평균 득표율 {np.mean(shares):.3f}")
assert len(ids) == len(work), f"행 수 불일치: {len(ids)} vs {len(work)}"

sub = pd.DataFrame({"id": ids, "answer": [int(p) for p in preds]})
sub["answer"] = sub["answer"].astype("int64")
sub.to_csv(f"{WORK}/submission.csv", index=False)

print(f"\nsubmission.csv 저장: {sub.shape}")
print(open(f"{WORK}/submission.csv").read()[:150])

---
## [8] 최종 점검 ▶️

제출 전 자동 검사. 하나라도 실패하면 제출하지 마세요.

In [ ]:
ok = True
def check(cond, msg):
    global ok
    print(("  OK   " if cond else "  FAIL ") + msg)
    ok = ok and cond

print("제출 파일 점검")
check(list(sub.columns) == ["id", "answer"], "컬럼이 소문자 id, answer 순서")
check(len(sub) == len(work), f"행 수 {len(sub)} == 문제 수 {len(work)}")
check(sub["id"].is_unique, "id 중복 없음")
check(set(sub["id"]) == set(work["id"]), "id 집합이 원본과 일치")
check(sub["answer"].notna().all(), "빈 값 없음")
check(str(sub["answer"].dtype) == "int64", f"answer 자료형 int64 (현재 {sub['answer'].dtype})")
check(n_fail/n_tot < 0.10, f"파싱 실패율 {n_fail/n_tot:.2%} < 10%")

print("\n" + ("전체 통과 — 제출 가능" if ok else "★ 실패 항목 있음 — 제출 금지"))

from IPython.display import FileLink
FileLink("submission.csv")

---
## [9] 제출 시 명시할 내용 ▶️ (규칙 5.2c)

**이걸 빠뜨리면 규칙 위반입니다.** 제출 양식에 그대로 옮기세요.

In [ ]:
print("""
[사용한 외부 데이터셋]
- AI-MO/NuminaMath-1.5 (Apache 2.0)
  https://huggingface.co/datasets/AI-MO/NuminaMath-1.5
  용도: SFT 학습 데이터
  처리: math-word-problem 중 정수 답 문항만 필터링, 무작위 추출(source별 비율 유지),
        평가 문항(로컬 검증 300 + 리더보드 831)은 사전 제거

[모델 구성]
- 베이스: Qwen/Qwen2.5-3B-Instruct (변경 없음)
- LoRA 어댑터 1개 (r=64, QLoRA 4bit NF4로 학습)
- 추론: 베이스 32샘플 + 어댑터 32샘플 = 64표 다수결 (모든 문제에 균일 적용, 라우팅 없음)
- 코드 실행/도구 호출/인터넷 접속 없음
""")